# 3. The growth-plan challenge: Bessent's "333"

Unit: America's Debt Crisis. Concept level: **understand → analyze → challenge → capstone**.

Data: the course dataset lives in one deterministic DuckDB file, `../../data/analytics.duckdb`, seeded
from `seed/seed.sql` (identical inside the Docker CLI). Every number here is
stable: rerun any notebook years from now and it reproduces the same answers,
because the seed uses fixed anchors (the course video's figures) plus
deterministic noise -- never the random number generator.

Workflow: run cells top to bottom. A `# TASK` comment marks a cell you should
edit; answer in the markdown cell just below when asked.

The video's reform anchor is the **"333" plan**: 3% growth, 3% primary
deficit, 3M extra barrels of oil equivalent a day by 2028. This notebook
decomposes it into three levers and challenges each one -- the way the video
challenges the CBO's happy path. You leave with a verdict.

* Lever 1 -- **3% growth**: raises g so the crossover arrives sooner.
* Lever 2 -- **3% primary deficit**: the budget shape after interest.
* Lever 3 -- **3M boe/d** by 2028: the energy/revenue pivot.

The composite matters: 3% growth with a 5% primary deficit is still a plan
that loses the plot.

### 3.1 What the dataset says about each lever

In [ ]:
# Bootstrap
%matplotlib inline
import sys
from pathlib import Path
import duckdb

sys.path.insert(0, "../../scripts")   # the g-vs-r model used in 02-03
import model as m

DB = "../../data/analytics.duckdb"
con = duckdb.connect(DB)

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams.update({"figure.figsize": (8, 4.4),
                     "axes.grid": True, "grid.alpha": 0.35,
                     "font.size": 10})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

gq = con.execute(
    "SELECT quarter, real_gdp_growth_pct AS growth_pct_annualized "
    "FROM growth_quarters ORDER BY quarter"
).fetchdf()
q = pd.to_datetime(gq.quarter.str.replace("Q", "-"))
axes[0].plot(q, gq.growth_pct_annualized, marker="o", ms=3)
axes[0].axhline(3.0, color="C3", ls="--", lw=1)
axes[0].set_title("Real growth (annualized, %): the 3% huddle")
axes[0].tick_params(axis="x", rotation=60)

e = con.execute(
    "SELECT year, crude_mbpd, natural_gas_mboed, total_boe_mbpd "
    "FROM energy ORDER BY year"
).fetchdf()
axes[1].plot(e.year, e.total_boe_mbpd, marker="o", ms=3)
axes[1].axhline(18.7, color="C2", ls="--", lw=1)
axes[1].set(title="Total oil+gas (M boe/d) vs 18.7 target")
plt.tight_layout(); plt.show()

### 3.2 Run the 333 plan through the model

In [ ]:
plan = m.project(30, g2026=3.0)
plan_df = pd.DataFrame(plan)
c_plan = m.crossover(plan)
print(f"333-plan crossover: {c_plan}")
assert c_plan == 2026, "3% growth immediately outruns 2.5% r"

base = m.project(30)
fig, ax = plt.subplots()
ax.plot(pd.DataFrame(base).year, pd.DataFrame(base).g_minus_r,
        label="baseline g-r (cross 2031)", color="C0")
ax.plot(plan_df.year, plan_df.g_minus_r, label="333 plan g-r", color="C2")
ax.axhline(0, color="C3", ls="--")
ax.set(xlabel="year", ylabel="g - r (pp)", title="Does 3% growth buy early breathing room?")
ax.legend(); plt.show()

### 3.3 The primary-deficit leg

In [ ]:
primary = con.execute(
    "SELECT d.fiscal_year, "
    "  d.receipts_billions - (d.outlays_billions - COALESCE(i.net_interest_billions, 0)) "
    "    AS primary_surplus_billions "
    "FROM deficit d LEFT JOIN interest i ON i.year = d.fiscal_year "
    "WHERE d.fiscal_year >= 2000 ORDER BY d.fiscal_year"
).fetchdf()
gdp_map = con.execute("SELECT year, nominal_gdp_billions FROM gdp").fetchdf()
primary["pct_gdp"] = primary.primary_surplus_billions / (
    gdp_map.set_index("year").reindex(primary.fiscal_year).nominal_gdp_billions.values
)
fig, ax = plt.subplots()
ax.plot(primary.fiscal_year, primary.pct_gdp, marker="o", ms=3)
ax.axhline(0, color="C3")
ax.set(xlabel="fiscal year", ylabel="primary surplus, % GDP",
       title="Primary balance: the discipline lever")
plt.show()
print("2026 primary surplus:", round(primary.pct_gdp.iloc[-1] * 100, 2), "% of GDP")
assert abs(primary.pct_gdp.iloc[-1] * 100 + 3.0) < 1.0, "the 3% primary deficit"

### 3.4 The composite verdict

Pull the pieces together. **3% growth** hands the crossover far earlier
than the market-stressed baselines of notebook 2. **3% primary deficit** keeps
the debt-to-GDP slope flat even before interest falls. The **energy lever** is
the only one the executive branch can *fund* -- but it moves the slowest.

Verdict questions (write in the cell below):
1. Growth 3%: policy lever, or weather forecast?
2. Primary deficit 3%: does the data show it is achievable?  (Count `primary_surplus_flag` years.)
3. Energy 3M boe/d by 2028: is the 18.7M boe/d target met in our data?

In [ ]:
flags = con.execute(
    "SELECT SUM(primary_surplus_flag) AS years, COUNT(*) AS total FROM deficit"
).fetchdf()
print("Years of primary surplus in 1977-2026:",
      int(flags.years.iloc[0]), "of", int(flags.total.iloc[0]))
e28 = con.execute(
    "SELECT total_boe_mbpd FROM energy WHERE year = 2028"
).fetchdf().total_boe_mbpd.iloc[0]
print("2028 total production (target 18.7):", round(e28, 1))
assert int(flags.years.iloc[0]) == 12

---
End of notebook 3. Next: the open-ended **capstone project**.